Task 1: Generate DataFrame from CSV

In [ ]:
# Import
import findspark
findspark.init()

from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

# Create Spark Context and Session
sc = SparkContext.getOrCreate()
spark = SparkSession.builder.appName("Employee Analysis").getOrCreate()

# Now read CSV data
employees_df = spark.read.csv("employees.csv", header=True, inferSchema=True)
employees_df.show(5)

your 131072x1 screen size is bogus. expect trouble
25/11/17 19:47:36 WARN Utils: Your hostname, DESKTOP-5LUU2QE resolves to a loopback address: 127.0.1.1; using 172.30.52.101 instead (on interface eth0)
25/11/17 19:47:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/17 19:47:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+------+--------+------+---+----------+
|Emp_No|Emp_Name|Salary|Age|Department|
+------+--------+------+---+----------+
|   198|  Donald|  2600| 29|        IT|
|   199| Douglas|  2600| 34|     Sales|
|   200|Jennifer|  4400| 36| Marketing|
|   201| Michael| 13000| 32|        IT|
|   202|     Pat|  6000| 39|        HR|
+------+--------+------+---+----------+
only showing top 5 rows



Task 2: Define Schema for the Data

In [11]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define schema MATCHING the CSV column order
employee_schema = StructType([
    StructField("Emp_No", IntegerType(), nullable=False),
    StructField("Emp_Name", StringType(), nullable=False),
    StructField("Salary", IntegerType(), nullable=True),  # Salary BEFORE Age
    StructField("Age", IntegerType(), nullable=True),      # Age AFTER Salary
    StructField("Department", StringType(), nullable=True)
])

# Read with corrected schema
employees_df = spark.read.csv("employees.csv", header=True, schema=employee_schema)
employees_df.show(5)

+------+--------+------+---+----------+
|Emp_No|Emp_Name|Salary|Age|Department|
+------+--------+------+---+----------+
|   198|  Donald|  2600| 29|        IT|
|   199| Douglas|  2600| 34|     Sales|
|   200|Jennifer|  4400| 36| Marketing|
|   201| Michael| 13000| 32|        IT|
|   202|     Pat|  6000| 39|        HR|
+------+--------+------+---+----------+
only showing top 5 rows



Task 3: Display Schema of DataFram

In [12]:
employees_df.printSchema()

root
 |-- Emp_No: integer (nullable = true)
 |-- Emp_Name: string (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Department: string (nullable = true)



Task 4: Create a Temporary View

In [13]:
# Create a temporary view for SQL queries
employees_df.createOrReplaceTempView("employees")

 Task 5: Execute SQL Query (Age > 30)

In [14]:
result = spark.sql("SELECT * FROM employees WHERE Age > 30")
result.show()

+------+-----------+------+---+----------+
|Emp_No|   Emp_Name|Salary|Age|Department|
+------+-----------+------+---+----------+
|   199|    Douglas|  2600| 34|     Sales|
|   200|   Jennifer|  4400| 36| Marketing|
|   201|    Michael| 13000| 32|        IT|
|   202|        Pat|  6000| 39|        HR|
|   203|      Susan|  6500| 36| Marketing|
|   205|    Shelley| 12008| 33|   Finance|
|   206|    William|  8300| 37|        IT|
|   100|     Steven| 24000| 39|        IT|
|   102|        Lex| 17000| 37| Marketing|
|   103|  Alexander|  9000| 39| Marketing|
|   104|      Bruce|  6000| 38|        IT|
|   105|      David|  4800| 39|        IT|
|   106|      Valli|  4800| 38|     Sales|
|   107|      Diana|  4200| 35|     Sales|
|   109|     Daniel|  9000| 35|        HR|
|   110|       John|  8200| 31| Marketing|
|   111|     Ismael|  7700| 32|        IT|
|   112|Jose Manuel|  7800| 34|        HR|
|   113|       Luis|  6900| 34|     Sales|
|   116|     Shelli|  2900| 37|   Finance|
+------+---

Task 6: Average Salary by Department

In [15]:
query = """
SELECT Department, AVG(Salary) AS Avg_Salary
FROM employees
GROUP BY Department
"""
spark.sql(query).show()


+----------+-----------------+
|Department|       Avg_Salary|
+----------+-----------------+
|     Sales|5492.923076923077|
|        HR|           5837.5|
|   Finance|           5730.8|
| Marketing|6633.333333333333|
|        IT|           7400.0|
+----------+-----------------+



Task 7: Filter IT Department

In [17]:
from pyspark.sql.functions import col
it_employees = employees_df.filter(col("Department") == "IT")
it_employees.show()

+------+--------+------+---+----------+
|Emp_No|Emp_Name|Salary|Age|Department|
+------+--------+------+---+----------+
|   198|  Donald|  2600| 29|        IT|
|   201| Michael| 13000| 32|        IT|
|   206| William|  8300| 37|        IT|
|   100|  Steven| 24000| 39|        IT|
|   104|   Bruce|  6000| 38|        IT|
|   105|   David|  4800| 39|        IT|
|   111|  Ismael|  7700| 32|        IT|
|   129|   Laura|  3300| 38|        IT|
|   132|      TJ|  2100| 34|        IT|
|   136|   Hazel|  2200| 29|        IT|
+------+--------+------+---+----------+



Task 8: Add 10% Bonus to Salaries

In [19]:
employees_bonus = employees_df.withColumn(
    "SalaryAfterBonus", 
    (col("Salary") * 1.10).cast("integer")
)
employees_bonus.show()

+------+---------+------+---+----------+----------------+
|Emp_No| Emp_Name|Salary|Age|Department|SalaryAfterBonus|
+------+---------+------+---+----------+----------------+
|   198|   Donald|  2600| 29|        IT|            2860|
|   199|  Douglas|  2600| 34|     Sales|            2860|
|   200| Jennifer|  4400| 36| Marketing|            4840|
|   201|  Michael| 13000| 32|        IT|           14300|
|   202|      Pat|  6000| 39|        HR|            6600|
|   203|    Susan|  6500| 36| Marketing|            7150|
|   204|  Hermann| 10000| 29|   Finance|           11000|
|   205|  Shelley| 12008| 33|   Finance|           13208|
|   206|  William|  8300| 37|        IT|            9130|
|   100|   Steven| 24000| 39|        IT|           26400|
|   101|    Neena| 17000| 27|     Sales|           18700|
|   102|      Lex| 17000| 37| Marketing|           18700|
|   103|Alexander|  9000| 39| Marketing|            9900|
|   104|    Bruce|  6000| 38|        IT|            6600|
|   105|    Da

Task 9: Maximum Salary by Age

In [22]:
from pyspark.sql.functions import max

max_salary = employees_df.groupBy("Age").agg(max("Salary").alias("Max_Salary"))
max_salary.show()

+---+----------+
|Age|Max_Salary|
+---+----------+
| 31|      8200|
| 34|      7800|
| 28|     12008|
| 27|     17000|
| 26|      3600|
| 37|     17000|
| 35|      9000|
| 39|     24000|
| 38|      6000|
| 29|     10000|
| 32|     13000|
| 33|     12008|
| 30|      8000|
| 36|      7900|
+---+----------+



Task 10: Self-Join

In [23]:
emp1 = employees_df.alias("emp1")
emp2 = employees_df.alias("emp2")

self_join = emp1.join(emp2, col("emp1.Emp_No") == col("emp2.Emp_No"))
self_join.select("emp1.Emp_No", "emp1.Emp_Name", "emp1.Department").show()

+------+---------+----------+
|Emp_No| Emp_Name|Department|
+------+---------+----------+
|   198|   Donald|        IT|
|   199|  Douglas|     Sales|
|   200| Jennifer| Marketing|
|   201|  Michael|        IT|
|   202|      Pat|        HR|
|   203|    Susan| Marketing|
|   204|  Hermann|   Finance|
|   205|  Shelley|   Finance|
|   206|  William|        IT|
|   100|   Steven|        IT|
|   101|    Neena|     Sales|
|   102|      Lex| Marketing|
|   103|Alexander| Marketing|
|   104|    Bruce|        IT|
|   105|    David|        IT|
|   106|    Valli|     Sales|
|   107|    Diana|     Sales|
|   108|    Nancy|     Sales|
|   109|   Daniel|        HR|
|   110|     John| Marketing|
+------+---------+----------+
only showing top 20 rows



Task 11: Average Employee Age

In [27]:
from pyspark.sql.functions import avg

avg_age = employees_df.agg(avg("Age").alias("Average_Age"))
avg_age.show()


+-----------+
|Average_Age|
+-----------+
|      33.56|
+-----------+



 Task 12: Total Salary by Department

In [28]:
from pyspark.sql.functions import sum

total_salary = employees_df.groupBy("Department").agg(sum("Salary").alias("Total_Salary"))
total_salary.show()


+----------+------------+
|Department|Total_Salary|
+----------+------------+
|     Sales|       71408|
|        HR|       46700|
|   Finance|       57308|
| Marketing|       59700|
|        IT|       74000|
+----------+------------+



Task 13: Sort by Age and Salary

In [29]:
sorted_df = employees_df.orderBy(col("Age").asc(), col("Salary").desc())
sorted_df.show()

+------+---------+------+---+----------+
|Emp_No| Emp_Name|Salary|Age|Department|
+------+---------+------+---+----------+
|   137|   Renske|  3600| 26| Marketing|
|   101|    Neena| 17000| 27|     Sales|
|   114|      Den| 11000| 27|   Finance|
|   108|    Nancy| 12008| 28|     Sales|
|   130|    Mozhe|  2800| 28| Marketing|
|   126|    Irene|  2700| 28|        HR|
|   204|  Hermann| 10000| 29|   Finance|
|   115|Alexander|  3100| 29|   Finance|
|   134|  Michael|  2900| 29|     Sales|
|   198|   Donald|  2600| 29|        IT|
|   140|   Joshua|  2500| 29|   Finance|
|   136|    Hazel|  2200| 29|        IT|
|   120|  Matthew|  8000| 30|        HR|
|   110|     John|  8200| 31| Marketing|
|   127|    James|  2400| 31|        HR|
|   201|  Michael| 13000| 32|        IT|
|   111|   Ismael|  7700| 32|        IT|
|   119|    Karen|  2500| 32|   Finance|
|   205|  Shelley| 12008| 33|   Finance|
|   124|    Kevin|  5800| 33| Marketing|
+------+---------+------+---+----------+
only showing top

Task 14: Count Employees per Department

In [30]:
from pyspark.sql.functions import count

employee_count = employees_df.groupBy("Department").agg(count("*").alias("Employee_Count"))
employee_count.show()

+----------+--------------+
|Department|Employee_Count|
+----------+--------------+
|     Sales|            13|
|        HR|             8|
|   Finance|            10|
| Marketing|             9|
|        IT|            10|
+----------+--------------+

